# RUmo RAG — Entrega 3: Benchmark Comparativo em GPU (Google Colab)
### Avaliação de Qualidade e Precisão: LLM Puro (Sem RAG) vs. RUmo RAG + Reescrita de Consultas em Dois Estágios

Este notebook executa o pipeline oficial de avaliação empírica da **Entrega 3** do projeto **RUmo (Residência em TIC 20 - Grupo 4 / FCTE UnB)**.

---

### Objetivos da Avaliação
1. **Comparar lado a lado** as respostas geradas por LLM puro (*Sem RAG*) contra o assistente fundamentado (*Com RAG*).
2. **Avaliar a reescrita de consultas compostas em dois estágios:**
   - **Estágio 1 (Classificação rápida):** `LiquidAI/d1-omni-600M` em GPU para discriminar perguntas compostas/multi-hop.
   - **Estágio 2 (Decomposição contextual):** `Qwen/Qwen2.5-7B-Instruct` para gerar sub-consultas independentes com termos acadêmicos explícitos.
3. **Calcular métricas objetivas de qualidade e precisão (sem latência):**
   - **ALCE Valid Citation Rate** (taxa de afirmações com fontes válidas `[n]`).
   - **RAGAS Factual Faithfulness** (fidelidade das respostas aos documentos oficiais).
   - **Auditoria de Alucinação Numérica** (`valores_sem_fonte` em preços, horários, datas e códigos).
   - **RGB Negative Rejection** (capacidade de recusar responder perguntas fora do domínio).
   - **RGB Multi-hop Completeness** (completude de cobertura em perguntas compostas).
   - **ROUGE-L** (sobreposição lexical LCS com as respostas de referência).

> **Requisito R3 (Zero Latência):** Em total conformidade com as diretrizes da Entrega 3, métricas de tempo de execução, latência de inferência ou throughput foram estritamente excluídas da avaliação comparativa.

## 1. Inspeção de Hardware e GPU
Verifique se o runtime do Colab está configurado com GPU (T4, L4, A100 ou H100).
Recomendado: **L4 (24GB) ou A100 (40GB/80GB)** para execução com precisão total (FP16/BF16 sem quantização).

In [ ]:
# Inspeção dos recursos de GPU alocados no Colab
!nvidia-smi

import os
import torch

print("\n--- Verificação PyTorch / CUDA ---")
print(f"CUDA Disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_bytes = torch.cuda.get_device_properties(0).total_memory
    vram_gb = vram_bytes / (1024 ** 3)
    print(f"Dispositivo: {device_name}")
    print(f"VRAM Total: {vram_gb:.2f} GB")
    print(f"Suporte a bfloat16: {torch.cuda.is_bf16_supported()}")
    if vram_gb >= 20.0:
        print(">> GPU de alta capacidade detectada (A100/L4). Precisão total FP16/BF16 recomendada.")
    else:
        print(">> GPU padrão (T4). Quantização 4-bit (bitsandbytes) recomendada para otimizar VRAM.")
else:
    print("ALERTA: Nenhuma GPU ativa. Acesse Ambiente de execução -> Alterar tipo de ambiente e selecione uma GPU.")

## 2. Configuração de Autenticação (Opcional)
Se você tiver um token pessoal do Hugging Face (`HF_TOKEN`), adicione-o aos **Secrets** do Colab (ícone de chave na barra lateral) para evitar limites de taxa de download de modelos abertos.

In [ ]:
# Configuração opcional do Hugging Face Token via Colab Secrets
import os

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN") or userdata.get("HUGGING_FACE_HUB_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = token
        print("HF_TOKEN configurado com sucesso via Colab Secrets!")
    else:
        print("Nenhum HF_TOKEN encontrado em Secrets. Download anônimo será utilizado (funciona normalmente).")
except Exception as e:
    print(f"Secrets não disponíveis ({e}). Prosseguindo com download público.")

## 3. Clonagem do Repositório e Checkout da Branch
Clona o repositório oficial do projeto na branch de trabalho `feat/validacao-reescrita-consulta`.

In [ ]:
# Clone e acesso à branch da Entrega 3 (idempotente para re-execução)
from pathlib import Path
if not Path("colab_benchmark.py").exists():
    if not Path("residencia-grupo-4").exists():
        !git clone -b feat/validacao-reescrita-consulta https://github.com/maykonjuso/residencia-grupo-4.git
    %cd residencia-grupo-4
!git pull origin feat/validacao-reescrita-consulta
!git status

## 4. Instalação das Dependências do Projeto
Instala todas as dependências especificadas em `requirements.txt`.

In [ ]:
# Instalação dos pacotes necessários
!pip install -q -r requirements.txt

## 5. Execução Autônoma do Benchmark em GPU
O script `colab_benchmark.py`:
1. Detecta automaticamente a VRAM disponível e ajusta a quantização (`RUMO_HF_4BIT=0` para A100/L4, `1` para T4).
2. Verifica se a base vetorial ChromaDB já está indexada; se vazia, indexa automaticamente os 1.738 chunks em ~15s.
3. Pré-aquece os modelos `LiquidAI/d1-omni-600M` e `Qwen/Qwen2.5-7B-Instruct` na VRAM da GPU.
4. Executa a avaliação sobre as 38 perguntas (32 perguntas-ouro, 5 perguntas compostas e 1 pergunta de recusa).
5. Executa auditoria estrita de conformidade com o Requisito R3 (Zero Latência).
6. Salva as saídas consolidadas em `rag/avaliacao/resultado_benchmark_colab.json` e `.md`.

In [ ]:
# Executa o benchmark com backend Hugging Face local na GPU
!python colab_benchmark.py --backend hf --saida md --arquivo-saida rag/avaliacao/resultado_benchmark_colab.json

## 6. Visualização e Análise dos Resultados
Carrega e renderiza o relatório Markdown gerado e a tabela de métricas consolidadas.

In [ ]:
# Renderização do relatório consolidado e tabela comparativa
import json
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd

md_path = Path("rag/avaliacao/resultado_benchmark_colab.md")
json_path = Path("rag/avaliacao/resultado_benchmark_colab.json")

if md_path.exists():
    display(Markdown(md_path.read_text(encoding="utf-8")))

if json_path.exists():
    dados = json.loads(json_path.read_text(encoding="utf-8"))
    metricas = dados.get("metricas_consolidadas", {})
    sem = metricas.get("sem_rag", {})
    com = metricas.get("com_rag", {})
    
    tabela_df = pd.DataFrame([
        {"Métrica": "Taxa de Citação Válida (ALCE)", "Sem RAG (LLM Puro)": f"{sem.get('taxa_citacao_valida', 0):.2%}", "Com RAG (RUmo Híbrido)": f"{com.get('taxa_citacao_valida', 0):.2%}", "Meta": "≥ 95%"},
        {"Métrica": "Fidelidade Factual (RAGAS)", "Sem RAG (LLM Puro)": f"{sem.get('fidelidade_factual', 0):.2%}", "Com RAG (RUmo Híbrido)": f"{com.get('fidelidade_factual', 0):.2%}", "Meta": "≥ 90%"},
        {"Métrica": "Taxa de Alucinação (Valores sem fonte)", "Sem RAG (LLM Puro)": f"{sem.get('taxa_alucinacao_numerica', 0):.2%}", "Com RAG (RUmo Híbrido)": f"{com.get('taxa_alucinacao_numerica', 0):.2%}", "Meta": "0.00%"},
        {"Métrica": "Rejeição Negativa / Recusa (RGB)", "Sem RAG (LLM Puro)": f"{sem.get('taxa_recusa_negativa', 0):.2%}", "Com RAG (RUmo Híbrido)": f"{com.get('taxa_recusa_negativa', 0):.2%}", "Meta": "100%"},
        {"Métrica": "Completude Multi-hop (RGB)", "Sem RAG (LLM Puro)": f"{sem.get('completude_multihop', 0):.2%}", "Com RAG (RUmo Híbrido)": f"{com.get('completude_multihop', 0):.2%}", "Meta": "≥ 85%"},
        {"Métrica": "ROUGE-L (F1 Médio)", "Sem RAG (LLM Puro)": f"{sem.get('rouge_l', 0):.2f}", "Com RAG (RUmo Híbrido)": f"{com.get('rouge_l', 0):.2f}", "Meta": "Informativa"},
    ])
    display(tabela_df)

## 7. Persistência dos Resultados

Abaixo são disponibilizadas três opções para exportar e persistir os resultados gerados (`resultado_benchmark_colab.json`, `resultado_benchmark_colab.md`, `respostas_comparativas.json` e `resultado_fidelidade.json`).

### Opção A: Download Direto para o Computador (Padrão e Seguro)
Baixa os arquivos de resultado diretamente pelo navegador.

In [ ]:
# Download direto dos arquivos de resultado via navegador
from google.colab import files

arquivos_para_download = [
    "rag/avaliacao/resultado_benchmark_colab.json",
    "rag/avaliacao/resultado_benchmark_colab.md",
    "rag/avaliacao/respostas_comparativas.json",
    "rag/avaliacao/resultado_fidelidade.json",
]

for arq in arquivos_para_download:
    try:
        files.download(arq)
        print(f"Download iniciado: {arq}")
    except Exception as e:
        print(f"Não foi possível baixar {arq}: {e}")

### Opção B: Cópia para o Google Drive
Monta o Google Drive pessoal e copia os arquivos para uma pasta dedicada.

In [ ]:
# Salvar no Google Drive (em navegador Colab)
import shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    destino_drive = Path("/content/drive/MyDrive/rumo_rag_entrega3_resultados")
    destino_drive.mkdir(parents=True, exist_ok=True)
    for arq in [
        "rag/avaliacao/resultado_benchmark_colab.json",
        "rag/avaliacao/resultado_benchmark_colab.md",
        "rag/avaliacao/respostas_comparativas.json",
        "rag/avaliacao/resultado_fidelidade.json",
    ]:
        p = Path(arq)
        if p.exists():
            shutil.copy(p, destino_drive / p.name)
            print(f"Salvo no Drive: {destino_drive / p.name}")
except Exception as e:
    print(f"Montagem do Drive ignorada (ex.: execução via extensão VS Code): {e}")

### Opção C (Opcional): Commit Direto no Repositório GitHub
Caso você possua um Personal Access Token do GitHub configurado nos Secrets do Colab como `GITHUB_TOKEN` com permissão de escrita (`repo`), você pode commitar os resultados diretamente na branch `feat/validacao-reescrita-consulta`.

In [ ]:
# Commit direto no GitHub usando token dos Secrets (Opcional)
import os

try:
    from google.colab import userdata
    gh_token = userdata.get("GITHUB_TOKEN")
    if gh_token:
        !git config --global user.name "Colab GPU Benchmark Runner"
        !git config --global user.email "colab-runner@fcte.unb.br"
        !git add rag/avaliacao/resultado_benchmark_colab.* rag/avaliacao/respostas_comparativas.json rag/avaliacao/resultado_fidelidade.json
        !git commit -m "docs(benchmark): add Colab GPU benchmark results and comparative answers [skip ci]"
        !git push https://{gh_token}@github.com/maykonjuso/residencia-grupo-4.git feat/validacao-reescrita-consulta
        print("Resultados commitados e enviados para a branch feat/validacao-reescrita-consulta com sucesso!")
    else:
        print("GITHUB_TOKEN não configurado em Secrets. Use a Opção A (download direto) ou Opção B (Drive).")
except Exception as e:
    print(f"Instrução opcional ignorada: {e}")